In [2]:
import sys
import os 

import numpy as np 
import pandas as pd 

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

sys.path.append(os.path.abspath("../.."))
from src.results_manager import ResultsManager

rm = ResultsManager("logistic_regression")

In [2]:
from src.load_processed import load_processed_data

dataset = load_processed_data("../../data/processed/preprocessed.npz")

X_train = dataset["X_train"]
X_test = dataset["X_test"]

y_train = dataset["y_train"]
y_test = dataset["y_test"]

font_test = dataset["font_test"]
italic_test = dataset["italic_test"]
strength_test = dataset["strength_test"]

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("Number of classes:", len(np.unique(y_train)))
print("Data type:", X_train.dtype)

Train shape: (666136, 400)
Test shape: (166534, 400)
Number of classes: 256
Data type: float32


In [3]:
model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        solver = "saga",
        max_iter = 35,
        tol = 1e-3,
        n_jobs = -1,
        verbose = 1,
        C = 1.0,
        random_state = 42
    ))
])

Uputstvo: 
model mozete trenirati ispocetka (dug proces) ili koristiti vec istreniran i skladisten model. <br>Ukoliko zelite da trenirate model ponovo, izbrisite ga iz results/logistic_regression/model.pkl. <br>
Inace, samo pokrenite celiju, ona ce ucitati istrenirani model.

In [4]:
import time 

trained_now = False 
training_time = None

model_loaded = rm.load_model()

if model_loaded is not None: 
    model = model_loaded
    print("Successfully loaded existing model.")
else:
    print("Training model ... ")
    start_time = time.time()

    model.fit(X_train, y_train)

    training_time = time.time() - start_time
    print(f"Training done in {training_time:.2f} seconds")

    # save model using results manager 
    rm.save_model(model)

    trained_now = True 

Path koju gledam: ../../results/logistic_regression/model.pkl
Training model ... 


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.


Epoch 1, change: 1
Epoch 2, change: 0.25309759
Epoch 3, change: 0.13164525
Epoch 4, change: 0.10595068
Epoch 5, change: 0.088798895
Epoch 6, change: 0.076358937
Epoch 7, change: 0.071859963
Epoch 8, change: 0.04170502
Epoch 9, change: 0.038670804
Epoch 10, change: 0.037665386
Epoch 11, change: 0.037063755
Epoch 12, change: 0.038279723
Epoch 13, change: 0.038366798
Epoch 14, change: 0.03642302
Epoch 15, change: 0.035986163
Epoch 16, change: 0.036855079
Epoch 17, change: 0.031671584
Epoch 18, change: 0.019503856
Epoch 19, change: 0.018546175
Epoch 20, change: 0.018593727
Epoch 21, change: 0.018452952
Epoch 22, change: 0.016916769
Epoch 23, change: 0.01015813
Epoch 24, change: 0.0096225021
Epoch 25, change: 0.0095628761
Epoch 26, change: 0.0094925268
Epoch 27, change: 0.0094295777
Epoch 28, change: 0.0047665602
Epoch 29, change: 0.0047375024
Epoch 30, change: 0.0046831081
Epoch 31, change: 0.0046702451
Epoch 32, change: 0.004531384
Epoch 33, change: 0.0045447815
Epoch 34, change: 0.004341

/home/nemanja/.local/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [5]:

y_pred = model.predict(X_test)

In [6]:
from sklearn.metrics import accuracy_score, f1_score

accuracy = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average = "macro")
weighted_f1 = f1_score(y_test, y_pred, average = "weighted")

print("Accuracy:", accuracy)
print("Macro F1:", macro_f1)
print("Weighted F1:", weighted_f1)

Accuracy: 0.4776802334658388
Macro F1: 0.3363497733267168
Weighted F1: 0.45624224487458903


In [7]:
import json 

if training_time is None:
    training_time = -0.1

results = {
    "model": "logistic_regression",
    "accuracy": float(accuracy),
    "macro_f1": float(macro_f1),
    "weighted_f1": float(weighted_f1),
    "training_time_seconds": float(training_time)
}

# save results in .json file only if model is trained now: 
if trained_now:
    rm.save_metrics(results)


Metrics saved to ../../results/logistic_regression/metrics.json


In [8]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print(f"Shape: {cm.shape}\n")
print(cm)

Shape: (256, 256)

[[ 45   1   2 ...   0   0   0]
 [  6  92   4 ...   0   0   0]
 [  2  36 156 ...   0   0   0]
 ...
 [  0   0   0 ... 137   0   0]
 [  0   1   1 ...   1 119   3]
 [  0   1   0 ...   0   2  50]]


In [9]:
errors = []

n = cm.shape[0]

for true_class in range(n):
    for pred_class in range(n):
        if true_class != pred_class:            # avoid diag elements 
            count = cm[true_class, pred_class]
            errors.append((true_class, pred_class, count))

errors = sorted(errors, key = lambda x: x[2], reverse = True)

top_errors = errors[:10]

top_errors_readable = []
for true_class, pred_class, count in top_errors:
    label = f"{chr(true_class)} → {chr(pred_class)}"
    top_errors_readable.append((label, count))

errors_df = pd.DataFrame(
    top_errors_readable,
    columns = ["confusion", "count"]
)

rm.save_dataframe(errors_df, "top_confusions")

errors_df

Dataframe saved to ../../results/logistic_regression/top_confusions.csv


,confusion,count
0,1 → /,309
1,2 → 1,249
2,O → 0,245
3,4 → 9,241
4,0 → 1,203
5,8 → 1,183
6,8 → 0,179
7,6 → 1,176
8,9 → 4,171
9,4 → 1,167


### Per-class accuracy 

In [10]:
# number of correct classifications per class (diag elements)
correct = np.diag(cm)

# total number of samples per class
total = cm.sum(axis = 1)

# per-class accuracy
per_class_accuracy = correct / total

# ASCII oznake klasa
labels = [f"{i}:{repr(chr(i))}" for i in range(cm.shape[0])]

# dataframe sa rezultatima
per_class_df = pd.DataFrame({
    "character": labels,
    "accuracy": per_class_accuracy,
    "correct": correct,
    "total": total,
    "percentage_%": per_class_accuracy *100
})

# sortiranje po tacnosti
per_class_df = per_class_df.sort_values("accuracy")
rm.save_dataframe(per_class_df, "class_accuracy")

worst_classified = per_class_df.head(10)
rm.save_dataframe(worst_classified, "worst_classified_classes")
print("Worst classified characters:")
print(worst_classified)

best_classified = per_class_df.tail(10)
rm.save_dataframe(best_classified, "best_classified_classes")
print("\nBest classified characters:")
print(best_classified)

Dataframe saved to ../../results/logistic_regression/class_accuracy.csv
Dataframe saved to ../../results/logistic_regression/worst_classified_classes.csv
Worst classified characters:
      character  accuracy  correct  total  percentage_%
153  153:'\x99'  0.005155        1    194      0.515464
151  151:'\x97'  0.030928        6    194      3.092784
8      8:'\x08'  0.037736       10    265      3.773585
159  159:'\x9f'  0.037838        7    185      3.783784
185     185:'¹'  0.038168       15    393      3.816794
23    23:'\x17'  0.043228       15    347      4.322767
127  127:'\x7f'  0.047244       12    254      4.724409
157  157:'\x9d'  0.049451        9    182      4.945055
135  135:'\x87'  0.050000       11    220      5.000000
16    16:'\x10'  0.050505       20    396      5.050505
Dataframe saved to ../../results/logistic_regression/best_classified_classes.csv

Best classified characters:
   character  accuracy  correct  total  percentage_%
78    78:'N'  0.711558      708    995

In [11]:
import warnings
warnings.filterwarnings("ignore")

font_results = []

for font in np.unique(font_test):
    mask = font_test == font

    acc = accuracy_score(
        y_test[mask],
        y_pred[mask]
    )

    font_results.append({
        "font": font,
        "samples": mask.sum(),
        "accuracy": acc
    })


font_df = pd.DataFrame(font_results)
font_df = font_df.sort_values("accuracy", ascending = False)
font_df["accuracy_%"] = (font_df["accuracy"] * 100).round(2)
rm.save_dataframe(font_df, "font_accuracy")

font_df

Dataframe saved to ../../results/logistic_regression/font_accuracy.csv


,font,samples,accuracy,accuracy_%
100,NUMERICS,2780,0.962950,96.29
43,E13B,4718,0.959093,95.91
102,OCRB,18706,0.905645,90.56
101,OCRA,13077,0.879330,87.93
91,MONEY,1499,0.856571,85.66
...,...,...,...,...
137,TAHOMA,2675,0.092710,9.27
18,BUXTON,483,0.091097,9.11
152,YI BAITI,1206,0.085406,8.54
126,SERIF,2606,0.079432,7.94


In [12]:
italic_results = []

for value in np.unique(italic_test):
    mask = italic_test == value 

    acc = accuracy_score(
        y_test[mask],
        y_pred[mask]
    )

    italic_results.append({
        "italic": value, 
        "samples": mask.sum(),
        "accuracy": acc
    })

italic_df = pd.DataFrame(italic_results)
italic_df["accuracy_%"] = (italic_df["accuracy"] * 100).round(2)
rm.save_dataframe(italic_df, "italic_accuracy")

italic_df

Dataframe saved to ../../results/logistic_regression/italic_accuracy.csv


,italic,samples,accuracy,accuracy_%
0,0,116206,0.571296,57.13
1,1,50328,0.261524,26.15


57.13% karaktera koji nisu italic je klasifikovano tacno  
26.15% karaktera koji jesu italic je klasifikovano tacno

In [13]:
bins = np.linspace(0, 1, 6)   # 0.0, 0.2, 0.4, 0.6, 0.8, 1.0
labels = [f"{bins[i]:.1f}-{bins[i+1]:.1f}" for i in range(len(bins)-1)]

strength_results = []

for i in range(len(bins)-1):

    mask = (strength_test >= bins[i]) & (strength_test < bins[i+1])

    samples = mask.sum()
    if samples == 0:
        acc = np.nan
    else: 
        acc = accuracy_score(
            y_test[mask],
            y_pred[mask]
        )


    strength_results.append({
        "strength_range": labels[i],
        "samples": samples,
        "accuracy": acc
    })


strength_df = pd.DataFrame(strength_results)
strength_df["accuracy_%"] = (strength_df["accuracy"] * 100).round(2)

rm.save_dataframe(strength_df, "strength_accuracy")

strength_df

Dataframe saved to ../../results/logistic_regression/strength_accuracy.csv


,strength_range,samples,accuracy,accuracy_%
0,0.0-0.2,0,NaN,NaN
1,0.2-0.4,0,NaN,NaN
2,0.4-0.6,114924,0.569037,56.90
3,0.6-0.8,51610,0.274249,27.42
4,0.8-1.0,0,NaN,NaN


Deblji karakteri imaju više spojenih piksela, pa se oblici slova međusobno približavaju.  
Na primer: 8 vs B, 5 vs S, O vs 0 -> jos slicniji kada je font deblji

### Logistic Regression - PCA 

In [3]:
from src.load_processed import load_processed_data

dataset = load_processed_data("../../data/processed/pca_preprocessed.npz")

X_train_pca = dataset["X_train"]
X_test_pca = dataset["X_test"]

y_train_pca = dataset["y_train"]
y_test_pca = dataset["y_test"]


print("Train shape:", X_train_pca.shape)
print("Test shape:", X_test_pca.shape)
print("Number of classes:", len(np.unique(y_train_pca)))
print("Data type:", X_train_pca.dtype)

Train shape: (666136, 119)
Test shape: (166534, 119)
Number of classes: 256
Data type: float32


In [4]:
model_pca = LogisticRegression(
        solver = "saga",
        max_iter = 35,
        tol = 1e-3,
        n_jobs = -1,
        verbose = 2,
        C = 1.0,
        random_state = 42
    )

In [5]:
import time 

print("Training model ... ")
start_time = time.time()

model_pca.fit(X_train_pca, y_train_pca)

training_time = time.time() - start_time
print(f"PCA Training done in {training_time:.2f} seconds")

Training model ... 


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.


Epoch 1, change: 1
Epoch 2, change: 0.22611475
Epoch 3, change: 0.10638576
Epoch 4, change: 0.070104405
Epoch 5, change: 0.062821217
Epoch 6, change: 0.046974149
Epoch 7, change: 0.049236756
Epoch 8, change: 0.037520837
Epoch 9, change: 0.017626751
Epoch 10, change: 0.013549481
Epoch 11, change: 0.010939654
Epoch 12, change: 0.0078933202
Epoch 13, change: 0.0050883903
convergence after 14 epochs took 856 seconds
PCA Training done in 855.80 seconds


In [6]:
y_pred_pca = model_pca.predict(X_test_pca)

In [7]:
from sklearn.metrics import accuracy_score, f1_score

accuracy_pca = accuracy_score(y_test_pca, y_pred_pca)
macro_f1_pca = f1_score(y_test_pca, y_pred_pca, average = "macro")
weighted_f1_pca = f1_score(y_test_pca, y_pred_pca, average = "weighted")

In [8]:
import json 

path = "../../results/logistic_regression/metrics.json"

with open(path, "r") as f:
    metrics = json.load(f)

accuracy = metrics["accuracy"]
macro_f1 = metrics["macro_f1"]
weighted_f1 = metrics["weighted_f1"]

print("Accuracy:", accuracy)
print("Macro F1:", macro_f1)
print("Weighted F1:", weighted_f1)


Accuracy: 0.4776802334658388
Macro F1: 0.3363497733267168
Weighted F1: 0.45624224487458903


In [9]:
comparison_df = pd.DataFrame({
    "Model": ["Logistic Regression", "Logistic Regression + PCA"],
    "accuracy": [accuracy, accuracy_pca],
    "Macro F1": [macro_f1, macro_f1_pca],
    "Weighted_F1": [weighted_f1, weighted_f1_pca]
})

rm.save_dataframe(comparison_df, "pca_comparison")

comparison_df

Dataframe saved to ../../results/logistic_regression/pca_comparison.csv


,Model,accuracy,Macro F1,Weighted_F1
0,Logistic Regression,0.47768,0.336350,0.456242
1,Logistic Regression + PCA,0.46265,0.313373,0.436223
